## RAG, step 1: the idea

**Problem:** the model does not know our documents and may invent an answer.

**Idea:** an open-book exam. Find the right page first, then answer from it.

| Part | Job | Uses the model? |
|---|---|---|
| 1. Keep | A small library of documents | No |
| 2. Find | Pick the best matching document(s) for the question | No (or a small embedding model) |
| 3. Answer | Give the model the question and the document; it answers from it | Yes |

**Why it gives a source:** we know which document we handed over.

**Keyword search by hand (documents D1 to D4, invented):**

| Question | Right note | Shared-word scores | Result |
|---|---|---|---|
| Which part carries current between breakers in a panel? | D2 | D2 scores 5, the others 1 or 0 | Works |
| How do I close the gap where a wire goes into a box? | D3 | All score 0 | Fails: different words, same meaning |

**Lesson:** matching on words breaks when the question uses different words. Step 2 compares meaning with embeddings.

## RAG, step 2a: embeddings

**Problem (from step 1):** matching shared words fails when the question uses different words for the same thing.

**Idea:** turn every sentence into a list of numbers that stands for its meaning. Sentences with similar meaning get similar lists. The computer then compares lists, not words.

| Word-matching | Embedding |
|---|---|
| Counts shared words | Compares meaning |
| "gap where a wire goes into a box" and "cable glands seal the point where a cable enters an enclosure" share almost no words, so the score is 0 | The two lists end up close together, so they match |

**A map picture:** think of a map of meaning. Every sentence is a pin. Sentences about the same thing land close together. A question is also dropped on the map, and the nearest pins are the best documents.

**Which model does this?** A separate, small model (`nomic-embed-text`). It does not write text. It only turns text into numbers.

**Next:** step 2b builds the first vector. Step 2c compares two vectors.


## RAG, step 2b: my first vector

| Item | My result |
|---|---|
| Embedding model | `nomic-embed-text` (small, separate from the text model) |
| Sentence | "Cable glands seal the point where a cable enters an enclosure." |
| Vector length | 768 |
| First five numbers | 0.0307, 0.0535, -0.1459, -0.0556, 0.0245 |
| My guess for the length | (fill in yours) |

| | Text model (`gemma4:e2b`) | Embedding model (`nomic-embed-text`) |
|---|---|---|
| Job | Writes text | Turns text into a position |
| Call | `ollama.chat(...)` | `ollama.embed(...)` |
| Result | `response.message.content` | `response.embeddings` (a list of vectors) |

```
response
└── embeddings    a list: one vector per input text
    └── [0]       the first vector, 768 numbers
```

**Reading it:** each number alone means nothing. The whole list, compared with other lists, carries the meaning.

## RAG, step 2c: compare two sentences

**Problem:** two sentences are now two lists of 768 numbers. We need one score for "how close in meaning".

**Idea:** cosine similarity. It compares the direction of the two lists and ignores their size.

| Code | Meaning |
|---|---|
| `ollama.embed(...)` | The action. Sends the sentences to the embedding model. |
| `.embeddings` | Not an action. It reads the list of vectors out of the reply, like `response.message.content` in a chat call. |
| `vectors[0]` | The vector of the first sentence |
| `a`, `b` | The two vectors being compared |
| `x`, `y` | One number from `a` and the matching number from `b` |
| `zip(a, b)` | Pairs the numbers up: first with first, second with second |
| `dot` | Multiply each pair, then add them up |
| `length_a`, `length_b` | The size of each vector: square every number, add them, take the square root |

**The formula in plain words:**

Score = dot ÷ (length_a × length_b)

**Tiny example (2 numbers each):**

| a | b | dot | length_a | length_b | Score |
|---|---|---|---|---|---|
| [1, 2] | [2, 4] | 10 | 2.236 | 4.472 | 1.0 (same direction) |
| [1, 0] | [0, 1] | 0 | 1 | 1 | 0 (unrelated) |

**How to read a score:** close to 1 means similar meaning. Close to 0 means unrelated.

**Mistake I made:** the function worked out `dot` and the two lengths but had no `return` line. It gave back `None`, and `round(None, 3)` raised a `TypeError`. A function without `return` hands back nothing.

**My results:**

| Pair | Sentences | Score |
|---|---|---|
| 0 vs 1 | Cable glands note and the question about the gap where a wire goes into a box | (fill in) |
| 0 vs 2 | Cable glands note and the busbar note | (fill in) |
| 1 vs 2 | The question and the busbar note | (fill in) |

**My guess for the highest pair:** (fill in)